# Predicting Heart Disease - S6E2
## v6: v4 + ビニングTE + カテゴリ組み合わせTE + 医学的指標

v4をベースに追加特徴量:
- ビニング（医学的閾値）→ Target Encoding
- カテゴリ組み合わせ → Target Encoding
- 医学的指標（RPP, Age²等）

In [5]:
import numpy as np
import pandas as pd
import gc
import warnings
from itertools import combinations

from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from scipy.stats import rankdata

warnings.filterwarnings('ignore')

In [6]:
class CFG:
    seed = 42
    n_splits = 5
    target_col = 'Heart Disease'
    use_original = True
    n_top_pairs = 8  # Feature Growthで使う交互作用ペア数

class Paths:
    p = '/Users/shirokoshikentaro/Desktop/Predicting Heart Disease/Data'
    train = p + '/train.csv'
    test = p + '/test.csv'
    sample = p + '/sample_submission.csv'
    original = p + '/dataset_heart.csv'

## 1. Data Loading + 原データ追加

In [7]:
train = pd.read_csv(Paths.train)
test = pd.read_csv(Paths.test)
sample_submission = pd.read_csv(Paths.sample)

if CFG.use_original:
    original = pd.read_csv(Paths.original)
    rename_map = {
        'age': 'Age', 'sex ': 'Sex',
        'chest pain type': 'Chest pain type',
        'resting blood pressure': 'BP',
        'serum cholestoral': 'Cholesterol',
        'fasting blood sugar': 'FBS over 120',
        'resting electrocardiographic results': 'EKG results',
        'max heart rate': 'Max HR',
        'exercise induced angina': 'Exercise angina',
        'oldpeak': 'ST depression',
        'ST segment': 'Slope of ST',
        'major vessels': 'Number of vessels fluro',
        'thal': 'Thallium',
        'heart disease': 'Heart Disease',
    }
    original = original.rename(columns=rename_map)
    original['Heart Disease'] = original['Heart Disease'].map({1: 0, 2: 1})
    if 'id' not in original.columns:
        max_id = train['id'].max()
        original['id'] = range(max_id + 1, max_id + 1 + len(original))
    original = original[train.columns]
    train = pd.concat([train, original], axis=0, ignore_index=True)
    cols_for_dedup = [c for c in train.columns if c != 'id']
    n_before = len(train)
    train = train.drop_duplicates(subset=cols_for_dedup, keep='first').reset_index(drop=True)
    print(f'After concat + dedup: {len(train)} (+{len(original)} orig, -{n_before - len(train)} dups)')

print(f'Train: {train.shape}, Test: {test.shape}')

After concat + dedup: 630270 (+270 orig, -0 dups)
Train: (630270, 15), Test: (270000, 14)


## 2. Target + Column Setup

In [8]:
TARGET = CFG.target_col
if train[TARGET].dtype == 'object' or train[TARGET].dtype.name == 'category':
    train[TARGET] = train[TARGET].map({'Absence': 0, 'Presence': 1})

y = train[TARGET].values.astype(np.uint8)
test_ids = test['id']

X_tr_raw = train.drop(columns=[TARGET, 'id'])
X_te_raw = test.drop(columns=['id'])

cat_cols = [
    'Sex', 'Chest pain type', 'FBS over 120', 'EKG results',
    'Exercise angina', 'Slope of ST', 'Number of vessels fluro', 'Thallium'
]
num_cols = ['Age', 'BP', 'Cholesterol', 'Max HR', 'ST depression']
all_base_cols = cat_cols + num_cols

# CV分割を固定（全セクションで共有）
skf_te = StratifiedKFold(n_splits=5, shuffle=True, random_state=CFG.seed)

print(f'Target: {np.unique(y)}, dist: 0={np.mean(y==0):.3f}, 1={np.mean(y==1):.3f}')

Target: [0 1], dist: 0=0.552, 1=0.448


## 3. Frequency Encoding

In [9]:
all_df = pd.concat([X_tr_raw, X_te_raw])
tr_freq = pd.DataFrame(index=X_tr_raw.index)
te_freq = pd.DataFrame(index=X_te_raw.index)
for c in all_base_cols:
    freq = all_df[c].value_counts(normalize=True)
    tr_freq[c + '_freq'] = X_tr_raw[c].map(freq)
    te_freq[c + '_freq'] = X_te_raw[c].map(freq)

print(f'Frequency features: {tr_freq.shape[1]}')

Frequency features: 13


## 4. Target Encoding（ベース13列）

In [10]:
tr_te = pd.DataFrame(index=X_tr_raw.index)
te_te = pd.DataFrame(index=X_te_raw.index)

for c in all_base_cols:
    tr_te[c + '_te'] = 0.0
    for tr_i, val_i in skf_te.split(X_tr_raw, y):
        means = train.iloc[tr_i].groupby(c)[TARGET].mean()
        tr_te.iloc[val_i, tr_te.columns.get_loc(c + '_te')] = X_tr_raw.iloc[val_i][c].map(means)
    te_te[c + '_te'] = X_te_raw[c].map(train.groupby(c)[TARGET].mean())

print(f'Base TE features: {tr_te.shape[1]}')

Base TE features: 13


## 5. ビニング → Target Encoding（NEW）
医学的に意味のある閾値で連続値をカテゴリ化し、TEを適用

In [11]:
# ビニング定義（医学的閾値）
bin_defs = {
    'Age_bin': ('Age', [0, 40, 50, 60, 70, 100]),
    'BP_bin': ('BP', [0, 120, 130, 140, 160, 300]),
    'Chol_bin': ('Cholesterol', [0, 200, 240, 280, 600]),
    'MaxHR_bin': ('Max HR', [0, 100, 120, 140, 160, 220]),
    'STdep_bin': ('ST depression', [-1, 0, 1, 2, 3, 10]),
}

# ビニングをtrain/testに適用
tr_bins = pd.DataFrame(index=X_tr_raw.index)
te_bins = pd.DataFrame(index=X_te_raw.index)

for bin_name, (col, bins) in bin_defs.items():
    tr_bins[bin_name] = pd.cut(X_tr_raw[col], bins=bins, labels=False).fillna(0).astype(int)
    te_bins[bin_name] = pd.cut(X_te_raw[col], bins=bins, labels=False).fillna(0).astype(int)

# ビニングにTarget Encoding適用
for bin_name in bin_defs.keys():
    te_col = bin_name + '_te'
    tr_te[te_col] = 0.0
    for tr_i, val_i in skf_te.split(X_tr_raw, y):
        means = pd.Series(y[tr_i], index=tr_bins.iloc[tr_i].index).groupby(tr_bins.iloc[tr_i][bin_name]).mean()
        tr_te.iloc[val_i, tr_te.columns.get_loc(te_col)] = tr_bins.iloc[val_i][bin_name].map(means).values
    full_means = pd.Series(y, index=tr_bins.index).groupby(tr_bins[bin_name]).mean()
    te_te[te_col] = te_bins[bin_name].map(full_means).values

# ビニングのFrequency Encodingも追加
all_bins = pd.concat([tr_bins, te_bins])
for bin_name in bin_defs.keys():
    freq = all_bins[bin_name].value_counts(normalize=True)
    tr_freq[bin_name + '_freq'] = tr_bins[bin_name].map(freq)
    te_freq[bin_name + '_freq'] = te_bins[bin_name].map(freq)

print(f'Binning features added: {len(bin_defs)} bins × (TE + Freq) = {len(bin_defs) * 2}')

Binning features added: 5 bins × (TE + Freq) = 10


## 6. カテゴリ組み合わせ → Target Encoding（NEW）
2カテゴリの組み合わせごとの疾病確率

In [12]:
cat_combos = [
    ('Sex', 'Chest pain type'),
    ('Thallium', 'Number of vessels fluro'),
    ('Exercise angina', 'Slope of ST'),
    ('Chest pain type', 'Exercise angina'),
    ('Sex', 'Exercise angina'),
    ('Thallium', 'Slope of ST'),
    ('Chest pain type', 'Thallium'),
    ('Sex', 'Thallium'),
]

for c1, c2 in cat_combos:
    combo_name = f'{c1}_{c2}_te'
    train_key = X_tr_raw[c1].astype(str) + '_' + X_tr_raw[c2].astype(str)
    test_key = X_te_raw[c1].astype(str) + '_' + X_te_raw[c2].astype(str)

    # OOF Target Encoding
    tr_te[combo_name] = 0.0
    for tr_i, val_i in skf_te.split(X_tr_raw, y):
        train_combo = X_tr_raw.iloc[tr_i][c1].astype(str) + '_' + X_tr_raw.iloc[tr_i][c2].astype(str)
        val_combo = X_tr_raw.iloc[val_i][c1].astype(str) + '_' + X_tr_raw.iloc[val_i][c2].astype(str)
        means = pd.Series(y[tr_i], index=train_combo.index).groupby(train_combo).mean()
        tr_te.iloc[val_i, tr_te.columns.get_loc(combo_name)] = val_combo.map(means).values

    full_combo = X_tr_raw[c1].astype(str) + '_' + X_tr_raw[c2].astype(str)
    full_means = pd.Series(y, index=full_combo.index).groupby(full_combo).mean()
    te_te[combo_name] = test_key.map(full_means).values

    # Frequency Encodingも追加
    all_key = pd.concat([train_key, test_key])
    freq = all_key.value_counts(normalize=True)
    tr_freq[f'{c1}_{c2}_freq'] = train_key.map(freq)
    te_freq[f'{c1}_{c2}_freq'] = test_key.map(freq)

print(f'Combo features added: {len(cat_combos)} combos × (TE + Freq) = {len(cat_combos) * 2}')

Combo features added: 8 combos × (TE + Freq) = 16


## 7. Feature Growth（Apriori-lite）

In [13]:
base = tr_te.fillna(0)
corr_scores = {}
for a, b in combinations(base.columns, 2):
    corr_scores[(a, b)] = abs(np.corrcoef(base[a], base[b])[0, 1])

top_pairs = sorted(corr_scores, key=corr_scores.get, reverse=True)[:CFG.n_top_pairs]

print(f'Top {CFG.n_top_pairs} correlated TE pairs:')
for a, b in top_pairs:
    print(f'  {a} × {b} (corr={corr_scores[(a,b)]:.4f})')
    tr_te[f'{a}_x_{b}'] = tr_te[a] * tr_te[b]
    te_te[f'{a}_x_{b}'] = te_te[a] * te_te[b]

print(f'\nTE features after Feature Growth: {tr_te.shape[1]}')

Top 8 correlated TE pairs:
  ST depression_te × STdep_bin_te (corr=0.9763)
  Thallium_te × Sex_Thallium_te (corr=0.9571)
  Thallium_te × Thallium_Slope of ST_te (corr=0.9215)
  Max HR_te × MaxHR_bin_te (corr=0.9139)
  Thallium_te × Thallium_Number of vessels fluro_te (corr=0.9077)
  Chest pain type_te × Sex_Chest pain type_te (corr=0.9064)
  Thallium_Slope of ST_te × Sex_Thallium_te (corr=0.8935)
  Thallium_Number of vessels fluro_te × Sex_Thallium_te (corr=0.8819)

TE features after Feature Growth: 34


## 8. 手動交互作用 + 医学的指標

In [14]:
def add_manual_features(df):
    out = pd.DataFrame(index=df.index)
    for col in df.select_dtypes(include=['int8', 'int16']).columns:
        df[col] = df[col].astype('int32')
    for col in df.select_dtypes(include=['float16']).columns:
        df[col] = df[col].astype('float32')

    # --- v4と同じ交互作用 ---
    out['Age_x_MaxHR'] = df['Age'] * df['Max HR']
    out['Age_div_MaxHR'] = df['Age'] / (df['Max HR'] + 1e-5)
    out['STdep_x_MaxHR'] = df['ST depression'] * df['Max HR']
    out['BP_x_Chol'] = df['BP'] * df['Cholesterol']
    out['Age_x_STdep'] = df['Age'] * df['ST depression']
    out['Age_x_Vessels'] = df['Age'] * df['Number of vessels fluro']
    out['BP_x_MaxHR'] = df['BP'] * df['Max HR']
    out['Chol_x_Age'] = df['Cholesterol'] * df['Age']
    out['STdep_x_Vessels'] = df['ST depression'] * df['Number of vessels fluro']
    out['Sex_x_MaxHR'] = df['Sex'] * df['Max HR']
    out['Sex_x_Chol'] = df['Sex'] * df['Cholesterol']
    out['ChestPain_x_STdep'] = df['Chest pain type'] * df['ST depression']
    out['ChestPain_x_MaxHR'] = df['Chest pain type'] * df['Max HR']
    out['ExAngina_x_STdep'] = df['Exercise angina'] * df['ST depression']
    out['ExAngina_x_MaxHR'] = df['Exercise angina'] * df['Max HR']
    out['Thallium_x_Vessels'] = df['Thallium'] * df['Number of vessels fluro']
    out['SlopeSTx_STdep'] = df['Slope of ST'] * df['ST depression']
    out['MaxHR_reserve'] = (220 - df['Age']) - df['Max HR']
    out['MaxHR_ratio'] = df['Max HR'] / (220 - df['Age'] + 1e-5)

    # --- NEW: 医学的指標 ---
    out['Rate_Pressure_Product'] = df['Max HR'] * df['BP']  # 心筋酸素消費量
    out['Chol_BP_ratio'] = df['Cholesterol'] / (df['BP'] + 1e-5)
    out['Age_squared'] = df['Age'] ** 2  # 加齢の非線形リスク
    out['STdep_squared'] = df['ST depression'] ** 2
    out['Chol_MaxHR_ratio'] = df['Cholesterol'] / (df['Max HR'] + 1e-5)

    return out

tr_manual = add_manual_features(X_tr_raw)
te_manual = add_manual_features(X_te_raw)
print(f'Manual features: {tr_manual.shape[1]} (v4: 19 + NEW: 5)')

Manual features: 24 (v4: 19 + NEW: 5)


## 9. Final Feature Set

In [15]:
X_train = pd.concat([X_tr_raw, tr_freq, tr_te, tr_manual], axis=1).fillna(0)
X_test = pd.concat([X_te_raw, te_freq, te_te, te_manual], axis=1).fillna(0)

print(f'Final feature set:')
print(f'  X_train: {X_train.shape}')
print(f'  X_test:  {X_test.shape}')
print(f'\nBreakdown:')
print(f'  Raw:               {X_tr_raw.shape[1]}')
print(f'  Freq Encoding:     {tr_freq.shape[1]}')
print(f'  Target Encoding:   {tr_te.shape[1]}')
print(f'  Manual + Medical:  {tr_manual.shape[1]}')
print(f'  Total:             {X_train.shape[1]}')

# 全てfloatか確認
non_numeric = X_train.select_dtypes(exclude=[np.number]).columns.tolist()
if non_numeric:
    print(f'\n⚠️ Non-numeric columns found: {non_numeric}')
    print('  → These will cause CatBoost errors!')
else:
    print(f'\n✅ All columns are numeric')

Final feature set:
  X_train: (630270, 97)
  X_test:  (270000, 97)

Breakdown:
  Raw:               13
  Freq Encoding:     26
  Target Encoding:   34
  Manual + Medical:  24
  Total:             97

✅ All columns are numeric


## 10. Model Parameters

In [16]:
cat_params = {
    'iterations': 3000,
    'learning_rate': 0.03,
    'depth': 4,
    'loss_function': 'Logloss',
    'eval_metric': 'AUC',
    'auto_class_weights': 'Balanced',
    'subsample': 0.65,
    'l2_leaf_reg': 12,
    'random_strength': 1.2,
    'bootstrap_type': 'Bernoulli',
    'task_type': 'CPU',
    'verbose': False,
}

xgb_params = {
    'n_estimators': 2500,
    'learning_rate': 0.03,
    'max_depth': 4,
    'subsample': 0.65,
    'colsample_bytree': 0.8,
    'reg_lambda': 12,
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'tree_method': 'hist',
    'verbosity': 0,
}

rf_params = {
    'n_estimators': 400,
    'max_depth': 6,
    'min_samples_leaf': 50,
    'class_weight': 'balanced',
    'n_jobs': -1,
    'random_state': CFG.seed,
}

print('Model parameters configured!')

Model parameters configured!


## 11. 5-Fold Training + Rank Stack

In [17]:
skf = StratifiedKFold(n_splits=CFG.n_splits, shuffle=True, random_state=CFG.seed)

oof_cb = np.zeros(len(X_train))
oof_xgb = np.zeros(len(X_train))
oof_rank = np.zeros(len(X_train))

test_cb = np.zeros(len(X_test))
test_xgb_pred = np.zeros(len(X_test))
test_rank = np.zeros(len(X_test))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y)):
    print(f'\n===== Fold {fold + 1} =====')

    # --- CatBoost ---
    cb = CatBoostClassifier(**cat_params, random_seed=CFG.seed + fold)
    cb.fit(X_train.iloc[tr_idx], y[tr_idx])
    val_cb = cb.predict_proba(X_train.iloc[val_idx])[:, 1]
    te_cb = cb.predict_proba(X_test)[:, 1]
    oof_cb[val_idx] = val_cb
    test_cb += te_cb / CFG.n_splits
    print(f'  CatBoost AUC: {roc_auc_score(y[val_idx], val_cb):.6f}')

    # --- XGBoost ---
    xgb_model = XGBClassifier(**xgb_params, random_state=CFG.seed + fold)
    xgb_model.fit(X_train.iloc[tr_idx], y[tr_idx])
    val_xgb = xgb_model.predict_proba(X_train.iloc[val_idx])[:, 1]
    te_xgb = xgb_model.predict_proba(X_test)[:, 1]
    oof_xgb[val_idx] = val_xgb
    test_xgb_pred += te_xgb / CFG.n_splits
    print(f'  XGBoost  AUC: {roc_auc_score(y[val_idx], val_xgb):.6f}')

    # --- Rank Average ---
    val_rank = (rankdata(val_cb) + rankdata(val_xgb)) / 2 / len(val_cb)
    te_rank = (rankdata(te_cb) + rankdata(te_xgb)) / 2 / len(te_cb)
    oof_rank[val_idx] = val_rank
    test_rank += te_rank / CFG.n_splits
    print(f'  Rank Avg AUC: {roc_auc_score(y[val_idx], val_rank):.6f}')

    gc.collect()

print('\n' + '=' * 50)
print(f'CatBoost OOF AUC: {roc_auc_score(y, oof_cb):.6f}')
print(f'XGBoost  OOF AUC: {roc_auc_score(y, oof_xgb):.6f}')
print(f'Rank Avg OOF AUC: {roc_auc_score(y, oof_rank):.6f}')


===== Fold 1 =====
  CatBoost AUC: 0.955824
  XGBoost  AUC: 0.955465
  Rank Avg AUC: 0.955752

===== Fold 2 =====
  CatBoost AUC: 0.955733
  XGBoost  AUC: 0.954486
  Rank Avg AUC: 0.955371

===== Fold 3 =====
  CatBoost AUC: 0.955009
  XGBoost  AUC: 0.954241
  Rank Avg AUC: 0.954807

===== Fold 4 =====
  CatBoost AUC: 0.955106
  XGBoost  AUC: 0.954416
  Rank Avg AUC: 0.954910

===== Fold 5 =====
  CatBoost AUC: 0.955352
  XGBoost  AUC: 0.954529
  Rank Avg AUC: 0.955145

CatBoost OOF AUC: 0.955400
XGBoost  OOF AUC: 0.954516
Rank Avg OOF AUC: 0.955197


## 12. Residual Random Forest

In [18]:
rf = RandomForestClassifier(**rf_params)
rf.fit(oof_rank.reshape(-1, 1), y)

rf_oof_pred = rf.predict_proba(oof_rank.reshape(-1, 1))[:, 1]
rf_test_pred = rf.predict_proba(test_rank.reshape(-1, 1))[:, 1]

best_alpha = 0.85
best_auc_blend = 0
for alpha in np.arange(0.5, 1.0, 0.05):
    blended = alpha * oof_rank + (1 - alpha) * rf_oof_pred
    auc = roc_auc_score(y, blended)
    if auc > best_auc_blend:
        best_auc_blend = auc
        best_alpha = alpha

print(f'Best alpha: {best_alpha:.2f}')
print(f'Blended OOF AUC: {best_auc_blend:.6f}')

final_test = best_alpha * test_rank + (1 - best_alpha) * rf_test_pred

Best alpha: 0.50
Blended OOF AUC: 0.955299


## 13. Submissions + Public Blend

In [25]:
# --- v6単体submissions ---
sub_main = pd.DataFrame({'id': test_ids, TARGET: final_test.astype(float)})
sub_main.to_csv('submission_v6_rank_rf.csv', index=False)
print('✅ submission_v6_rank_rf.csv')

sub_cb = pd.DataFrame({'id': test_ids, TARGET: test_cb.astype(float)})
sub_cb.to_csv('submission_v6_catboost.csv', index=False)
# public_sub = pd.read_csv('/sample_submission.csv')
print('✅ submission_v6_catboost.csv')

# # --- Public Notebook Blend ---
# def rank_avg_2(p1, p2, w1):
#     r1 = rankdata(p1) / len(p1)
#     r2 = rankdata(p2) / len(p2)
#     return w1 * r1 + (1 - w1) * r2

# public_sub = pd.read_csv(Paths.public_sub)
# public_pred = public_sub[TARGET].values

# print('\n--- Public Blends ---')
# for w in [0.1, 0.2, 0.3, 0.4, 0.5]:
#     blended = rank_avg_2(test_cb, public_pred, w)
#     sub = pd.DataFrame({'id': test_ids, TARGET: blended.astype(float)})
#     fname = f'submission_v6_blend_my{int(w*100)}_pub{int((1-w)*100)}.csv'
#     sub.to_csv(fname, index=False)
#     print(f'✅ {fname}')

✅ submission_v6_rank_rf.csv
✅ submission_v6_catboost.csv


In [26]:
print('=' * 60)
print('FINAL SUMMARY (v6)')
print('=' * 60)
print(f'  Features          : {X_train.shape[1]} columns')
print(f'    Raw:             {X_tr_raw.shape[1]}')
print(f'    Freq Encoding:   {tr_freq.shape[1]}')
print(f'    Target Encoding: {tr_te.shape[1]}')
print(f'    Manual+Medical:  {tr_manual.shape[1]}')
print(f'  CatBoost OOF AUC : {roc_auc_score(y, oof_cb):.6f}')
print(f'  XGBoost  OOF AUC : {roc_auc_score(y, oof_xgb):.6f}')
print(f'  Rank Avg OOF AUC : {roc_auc_score(y, oof_rank):.6f}')
print(f'  +RF Blend OOF AUC: {best_auc_blend:.6f}')
print('=' * 60)
print('\nv4 → v6 で追加した特徴量:')
print('  - ビニングTE (5種): Age, BP, Chol, MaxHR, STdep')
print('  - カテゴリ組み合わせTE (8種): Sex×ChestPain, Thal×Vessels, etc')
print('  - 医学的指標 (5種): RPP, Chol/BP, Age², STdep², Chol/MaxHR')
print('\n→ CatBoost単体のLBスコアをv4(0.95376)と比較してください')

FINAL SUMMARY (v6)
  Features          : 97 columns
    Raw:             13
    Freq Encoding:   26
    Target Encoding: 34
    Manual+Medical:  24
  CatBoost OOF AUC : 0.955400
  XGBoost  OOF AUC : 0.954516
  Rank Avg OOF AUC : 0.955197
  +RF Blend OOF AUC: 0.955299

v4 → v6 で追加した特徴量:
  - ビニングTE (5種): Age, BP, Chol, MaxHR, STdep
  - カテゴリ組み合わせTE (8種): Sex×ChestPain, Thal×Vessels, etc
  - 医学的指標 (5種): RPP, Chol/BP, Age², STdep², Chol/MaxHR

→ CatBoost単体のLBスコアをv4(0.95376)と比較してください
